# 🧪 Division 2 — STCR Engine (Soil Test Crop Response)
**Pipeline:** AI Precision Fertilizer System

**What happens here:**
1. Load `df_clean.pkl` from Division 1
2. Hardcode TNAU STCR equations for Paddy, Maize, Sugarcane, Groundnut (+ generic fallbacks)
3. Convert dataset soil NPK from mg/kg to kg/ha
4. Apply STCR formulas to compute `Target_N_kg_ha`, `Target_P_kg_ha`, `Target_K_kg_ha`
5. Validate: no negative doses, distribution checks
6. Save `df_stcr.pkl` with new STCR-derived labels for ML training

---
### STCR Formula Structure
```
Optimal_Dose = (Crop_Need * Target_Yield_quintals) - (Soil_Coeff * Soil_Nutrient_kg_ha) - (Organic_Coeff * Organic_Input)
```
For MVP: organic manure inputs (ON, OP, OK) = 0

In [ ]:
# Cell 1 | Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded')

In [ ]:
# Cell 2 | Load Clean Data
DATA_DIR = Path(r'D:\project sem 7\my-finalyear-project\UV\fertilizer\dats')
df = pd.read_pickle(DATA_DIR / 'df_clean.pkl')
print(f'Loaded df_clean.pkl: {df.shape}')
print(f'Crops: {sorted(df["Crop_Type"].unique())}')
df[['Crop_Type','Nitrogen_mg_kg','Phosphorus_mg_kg','Potassium_mg_kg','Target_Yield_ton_per_Hectare']].head()

## 🧮 Unit Conversion: mg/kg → kg/ha

The dataset has soil nutrients in **mg/kg**. STCR equations use **kg/ha**.

**Conversion:** `kg/ha = mg/kg * bulk_density(g/cm³) * depth(cm) / 10`

Standard assumptions:
- Bulk density = 1.3 g/cm³ (typical for Indian agricultural soils)
- Sampling depth = 15 cm (standard plough layer)

In [ ]:
# Cell 3 | Convert mg/kg to kg/ha
BULK_DENSITY = 1.3   # g/cm3 (typical Indian agricultural soil)
DEPTH_CM     = 15    # standard plough layer depth
CONV_FACTOR  = BULK_DENSITY * DEPTH_CM / 10  # = 1.95

print(f'Conversion factor: {CONV_FACTOR} (mg/kg -> kg/ha)')

df['Soil_N_kg_ha'] = df['Nitrogen_mg_kg']   * CONV_FACTOR
df['Soil_P_kg_ha'] = df['Phosphorus_mg_kg'] * CONV_FACTOR
df['Soil_K_kg_ha'] = df['Potassium_mg_kg']  * CONV_FACTOR

print('\nSoil NPK in kg/ha:')
for c in ['Soil_N_kg_ha','Soil_P_kg_ha','Soil_K_kg_ha']:
    print(f'  {c}: [{df[c].min():.1f}, {df[c].max():.1f}], mean={df[c].mean():.1f}')

## 📐 Yield Conversion: ton/ha → quintals/ha

STCR equations use **quintals/ha** (1 quintal = 100 kg = 0.1 ton).

`quintals = ton * 10`

In [ ]:
# Cell 4 | Convert yield to quintals
df['Target_Yield_q_ha'] = df['Target_Yield_ton_per_Hectare'] * 10
print(f'Target Yield (quintals/ha): [{df["Target_Yield_q_ha"].min():.1f}, {df["Target_Yield_q_ha"].max():.1f}]')
print(f'Mean: {df["Target_Yield_q_ha"].mean():.1f} q/ha')

## 🎯 TNAU STCR Equations (Hardcoded)

| Crop | FN (kg/ha) | FP₂O₅ (kg/ha) | FK₂O (kg/ha) |
|------|-----------|-------------|-------------|
| **Paddy** (Red Soil) | 3.49T - 0.36SN | 1.66T - 2.76SP | 2.19T - 0.66SK |
| **Maize** (Red Sandy Loam) | 3.96T - 0.62SN | 1.56T - 1.93SP | 1.66T - 0.27SK |
| **Sugarcane** (Alluvial/Red) | 4.18T - 0.58SN | 1.08T - 2.25SP | 3.24T - 0.66SK |
| **Groundnut** (Red Soil) | 4.10T - 0.42SN | 2.25T - 1.95SP | 3.10T - 0.50SK |

Where T = Target Yield (q/ha), SN/SP/SK = Soil N/P/K (kg/ha)

**Organic manure (ON/OP/OK) set to 0 for MVP.**

In [ ]:
# Cell 5 | Define STCR Equations

# TNAU STCR coefficients: {crop: {nutrient: (yield_coeff, soil_coeff)}}
# FN  = yield_coeff * T - soil_coeff * SN
# FP  = yield_coeff * T - soil_coeff * SP
# FK  = yield_coeff * T - soil_coeff * SK

STCR_COEFFS = {
    # Paddy / Rice - Red Soils
    'Rice':      {'N': (3.49, 0.36), 'P': (1.66, 2.76), 'K': (2.19, 0.66)},
    'Paddy':     {'N': (3.49, 0.36), 'P': (1.66, 2.76), 'K': (2.19, 0.66)},
    
    # Maize Hybrid - Red Sandy Loam
    'Maize':     {'N': (3.96, 0.62), 'P': (1.56, 1.93), 'K': (1.66, 0.27)},
    
    # Sugarcane - Alluvial / Red Soils
    'Sugarcane': {'N': (4.18, 0.58), 'P': (1.08, 2.25), 'K': (3.24, 0.66)},
    
    # Groundnut - Rainfed Red Soils (legume, lower N multiplier)
    'Groundnut': {'N': (4.10, 0.42), 'P': (2.25, 1.95), 'K': (3.10, 0.50)},
}

# Generic fallback for crops not in TNAU tables
# Based on averaged coefficients across all TNAU crops
STCR_GENERIC = {'N': (3.93, 0.50), 'P': (1.64, 2.22), 'K': (2.55, 0.52)}

def get_stcr_coeffs(crop_name):
    """Return STCR coefficients for a crop, falling back to generic if not found."""
    for key in STCR_COEFFS:
        if key.lower() in crop_name.lower():
            return STCR_COEFFS[key]
    return STCR_GENERIC

# Test
for crop in ['Paddy', 'Maize', 'Sugarcane', 'Groundnut', 'Tomato', 'Wheat']:
    c = get_stcr_coeffs(crop)
    print(f'{crop:12s} -> N({c["N"][0]}, {c["N"][1]})  P({c["P"][0]}, {c["P"][1]})  K({c["K"][0]}, {c["K"][1]})')

In [ ]:
# Cell 6 | Apply STCR Formulas to Entire Dataset

def compute_stcr(row):
    """Compute STCR-based fertilizer doses for a single row."""
    coeffs = get_stcr_coeffs(row['Crop_Type'])
    T  = row['Target_Yield_q_ha']
    SN = row['Soil_N_kg_ha']
    SP = row['Soil_P_kg_ha']
    SK = row['Soil_K_kg_ha']
    
    FN = coeffs['N'][0] * T - coeffs['N'][1] * SN
    FP = coeffs['P'][0] * T - coeffs['P'][1] * SP
    FK = coeffs['K'][0] * T - coeffs['K'][1] * SK
    
    return pd.Series({
        'Target_N_kg_ha': max(FN, 0),  # No negative fertilizer
        'Target_P_kg_ha': max(FP, 0),
        'Target_K_kg_ha': max(FK, 0)
    })

stcr_results = df.apply(compute_stcr, axis=1)
df = pd.concat([df, stcr_results], axis=1)

print('STCR targets computed!')
print('\nTarget NPK (kg/ha):')
for c in ['Target_N_kg_ha','Target_P_kg_ha','Target_K_kg_ha']:
    print(f'  {c}: [{df[c].min():.1f}, {df[c].max():.1f}], mean={df[c].mean():.1f}')

In [ ]:
# Cell 7 | Checkpoint: Inspect STCR output for each crop
print('=== STCR Target NPK by Crop (mean kg/ha) ===')
summary = df.groupby('Crop_Type')[['Target_N_kg_ha','Target_P_kg_ha','Target_K_kg_ha']].agg(['mean','std']).round(1)
print(summary)

# Sanity check: what % of rows have zero dose?
print('\n=== Zero-dose percentage ===')
for c in ['Target_N_kg_ha','Target_P_kg_ha','Target_K_kg_ha']:
    pct = (df[c] == 0).mean() * 100
    print(f'  {c}: {pct:.1f}% rows have zero dose')

In [ ]:
# Cell 8 | Visualise STCR Output Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
targets = ['Target_N_kg_ha','Target_P_kg_ha','Target_K_kg_ha']
colors  = ['#EF5350','#42A5F5','#66BB6A']
titles  = ['Nitrogen (N)','Phosphorus (P2O5)','Potassium (K2O)']

for ax, col, color, title in zip(axes, targets, colors, titles):
    ax.hist(df[col], bins=40, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean: {df[col].mean():.1f}')
    ax.set_title(f'STCR {title} Dose', fontweight='bold')
    ax.set_xlabel('kg/ha')
    ax.legend()

fig.suptitle('STCR-Derived Fertilizer Targets', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / 'stcr_targets_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9 | STCR Targets by Crop (boxplot)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, title in zip(axes, targets, titles):
    df.boxplot(column=col, by='Crop_Type', ax=ax)
    ax.set_title(f'{title} by Crop', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('kg/ha')
    ax.tick_params(axis='x', rotation=30)
fig.suptitle('STCR Doses by Crop', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / 'stcr_boxplot_by_crop.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 10 | Show example rows
sample_cols = ['Crop_Type','Target_Yield_q_ha','Soil_N_kg_ha','Soil_P_kg_ha','Soil_K_kg_ha',
               'Target_N_kg_ha','Target_P_kg_ha','Target_K_kg_ha']
print('=== Sample STCR Calculations ===')
df[sample_cols].sample(10, random_state=42)

## 💾 Save STCR-Enriched Dataset

In [ ]:
# Cell 11 | Save
df.to_pickle(DATA_DIR / 'df_stcr.pkl')
df.to_csv(DATA_DIR / 'df_stcr.csv', index=False)

print(f'Saved df_stcr.pkl: {df.shape}')
print(f'New columns added: Soil_N/P/K_kg_ha, Target_Yield_q_ha, Target_N/P/K_kg_ha')
print(f'\nTotal columns: {df.shape[1]}')
print('\nReady for Division 3 -> 03_ml_trainer.ipynb')